In [ ]:
# Install once if needed
# !pip install pdfplumber pandas openpyxl pymupdf

import pdfplumber
import pandas as pd
import re
from pathlib import Path
import fitz
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

# =========================
# EDIT YOUR PHRASES HERE
# =========================
PHRASES = [
    "Stayed one night",
    "Savannah, Georgia",
    "party",
    "company",
    "pool table",
    "projector",
    "ping pong",
    "pool",
    "fun",
    "bed bug",
    
]

pdf_folder = Path(".")
results = []

# =========================
# SCAN PDF TEXT
# =========================
results = []

# only include original PDFs, skip any previously highlighted ones
pdf_files = [f for f in pdf_folder.glob("*.pdf") if not f.stem.endswith("_HIGHLIGHTED")]

for pdf_file in pdf_files:
    with pdfplumber.open(pdf_file) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            if not text:
                continue

            lines = text.split("\n")

            for line_num, line in enumerate(lines, start=1):
                for phrase in PHRASES:
                    if re.search(re.escape(phrase), line, re.IGNORECASE):
                        # append " - Local Guest" if line is exactly "Savannah, Georgia"
                        line_text = line.strip()
                        if line_text.lower() == "savannah, georgia":
                            line_text += " - Local Guest"

                        results.append({
                            "file": pdf_file.name,
                            "page": page_num,
                            "line_number": line_num,
                            "phrase": phrase,
                            "line_text": line_text
                        })

df = pd.DataFrame(results)

# =========================
# SUMMARY TABLES
# =========================
phrase_summary = (
    df.groupby("phrase")
      .size()
      .reset_index(name="match_count")
      .sort_values("match_count", ascending=False)
)

location_summary = (
    df.groupby(["file", "page"])
      .size()
      .reset_index(name="matches_on_page")
      .sort_values(["file", "page"])
)

# =========================
# HIGHLIGHT MATCHES IN PDF
# =========================
for pdf_file in pdf_files:  # only highlight original PDFs
    doc = fitz.open(pdf_file)
    modified = False

    for page in doc:
        page_text = page.get_text("text")

        for phrase in PHRASES:
            pattern = re.compile(re.escape(phrase), re.IGNORECASE)

            for match in pattern.finditer(page_text):
                matched_text = match.group(0)
                text_instances = page.search_for(matched_text)

                for inst in text_instances:
                    highlight = page.add_highlight_annot(inst)
                    highlight.update()
                    modified = True

    if modified:
        output_name = pdf_file.stem + "_HIGHLIGHTED.pdf"
        doc.save(output_name, garbage=4, deflate=True)
    doc.close()

# =========================
# EXPORT EXCEL
# =========================
df.to_excel("airbnb_matches_detailed.xlsx", index=False)
phrase_summary.to_excel("airbnb_phrase_summary.xlsx", index=False)
location_summary.to_excel("airbnb_page_location_summary.xlsx", index=False)

# =========================
# GENERATE HTML REPORT
# =========================
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

html = f"""
<html>
<head>
<title>Airbnb Review Keyword Report</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 40px; }}
h1, h2 {{ color: #2c3e50; }}
table {{ border-collapse: collapse; width: 100%; margin-bottom: 30px; }}
th, td {{ border: 1px solid #ccc; padding: 8px; text-align: left; }}
th {{ background-color: #f2f2f2; }}
tr:nth-child(even) {{ background-color: #fafafa; }}
.summary-box {{
    padding: 15px;
    background: #f8f9fa;
    border: 1px solid #ddd;
    margin-bottom: 30px;
}}
</style>
</head>
<body>

<h1>Airbnb Review Keyword Report</h1>
<p><strong>Generated:</strong> {timestamp}</p>

<div class="summary-box">
<h2>Overview</h2>
<p><strong>Total Matches:</strong> {len(df)}</p>
<p><strong>Total PDFs Scanned:</strong> {len(pdf_files)}</p>
<p><strong>Phrases Searched:</strong> {", ".join(PHRASES)}</p>
</div>

<h2>Phrase Summary</h2>
{phrase_summary.to_html(index=False)}



<h2>Detailed Matches</h2>
{df.to_html(index=False)}

<h2>PDF Page Hotspots (Matches per Page)</h2>
{location_summary.to_html(index=False)}

</body>
</html>
"""

with open("airbnb_issue_report.html", "w", encoding="utf-8") as f:
    f.write(html)

# =========================
# DISPLAY IN NOTEBOOK
# =========================
print("HTML report saved as: airbnb_issue_report.html")
print("Open it in your browser → Print → Save as PDF")

display(phrase_summary)
display(location_summary)
display(df)

In [ ]:
# ============================================
# STR RENTAL ISSUE DETECTOR - Airbnb & VRBO
# Scans PDFs recursively for problematic keywords
# ============================================

# Install once if needed
# !pip install pdfplumber pandas openpyxl pymupdf

import pdfplumber
import pandas as pd
import re
from pathlib import Path
import fitz
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

# ============================================
# Using root words to catch variations
# ============================================

PHRASES = [
    # === PESTS ===
    "bed bug",
    "bedbug",
    "cockroach",
    "roach",
    "bug",
    "infestation",
    
    # === DIRT & FILTH (root words) ===
    "dirt",
    "dust",
    "filth",
    "floor",
    "sticky",
    "not cleaned",
    "cleaning issue",
    "trash",
    "garbage",
    "rock",
    "leaf",
    "unvacuumed",
    "stain",
    "freezer",
    "rug",
    
    # === GUEST-CREATED PROBLEMS (Parties/Groups) ===
    "bachelor",
    "bachelorette",
    "large group",
    "sleeps up to",
    "party",
    "graduation",
    "tournament",
    "super bowl",
    "baby shower",
    "entertainment home",
    "birthday",
    "joint bachelor",
    "group of",
    
    # === HOST MAINTENANCE ISSUES ===
    "run down",
    "tlc",
    "paint",
    "broken",
    "not working",
    "fire pit",
    "spa",
    "fridge",
    "cue ball",
    "drain",
    "standing water",
    "door sticks",
    "latch",
    "wankered",
    "ripped",
    "wall damage",
    "bed broken",
    "bathroom fixtures",
    "towel rack",
    
    # === HOST COMMUNICATION FAILURES ===
    "door code",
    "stuck in car",
    "voicemail",
    "delayed check-in",
    "coffee pod",
    "coffeepot",
    "grill",
    "putt putt",
    "listing accuracy",
    
    # === MISC ISSUES ===
    "asked if it would affect review",
    "hopeful i wouldn't leave a review",
    "quiet neighborhood",
]

# ============================================
# DIRECTORY SETUP
# ============================================

# Define the two main directories
AIRBNB_DIR = Path("AirBNB")
VRBO_DIR = Path("VRBO")

# Create output directories if they don't exist
AIRBNB_DIR.mkdir(exist_ok=True)
VRBO_DIR.mkdir(exist_ok=True)

# ============================================
# FUNCTION: SCAN PDFS IN A DIRECTORY RECURSIVELY
# ============================================

def scan_pdfs_in_directory(directory_path, results_list, source_name):
    """
    Recursively scan all PDFs in a directory and subdirectories.
    Highlights matches and returns results.
    """
    if not directory_path.exists():
        print(f"⚠️ Directory not found: {directory_path}")
        return
    
    # Find all PDF files recursively, skip previously highlighted ones
    pdf_files = list(directory_path.rglob("*.pdf"))
    pdf_files = [f for f in pdf_files if not f.stem.endswith("_HIGHLIGHTED")]
    
    print(f"\n📁 Scanning {source_name} directory: {directory_path}")
    print(f"   Found {len(pdf_files)} PDF(s) to process...")
    
    for pdf_file in pdf_files:
        print(f"   Processing: {pdf_file.name} (in {pdf_file.parent.name})")
        
        # Extract text and find matches
        with pdfplumber.open(pdf_file) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                text = page.extract_text()
                if not text:
                    continue
                
                lines = text.split("\n")
                
                for line_num, line in enumerate(lines, start=1):
                    for phrase in PHRASES:
                        if re.search(re.escape(phrase), line, re.IGNORECASE):
                            line_text = line.strip()
                            
                            # Flag local guests (Savannah residents)
                            if line_text.lower() == "savannah, georgia":
                                line_text += " - Local Guest"
                            
                            results.append({
                                "source": source_name,
                                "file": str(pdf_file.relative_to(directory_path.parent)) if pdf_file.parent != directory_path else pdf_file.name,
                                "page": page_num,
                                "line_number": line_num,
                                "phrase": phrase,
                                "line_text": line_text
                            })
        
        # Highlight matches in PDF
        doc = fitz.open(pdf_file)
        modified = False
        
        for page in doc:
            page_text = page.get_text("text")
            
            for phrase in PHRASES:
                pattern = re.compile(re.escape(phrase), re.IGNORECASE)
                
                for match in pattern.finditer(page_text):
                    matched_text = match.group(0)
                    text_instances = page.search_for(matched_text)
                    
                    for inst in text_instances:
                        highlight = page.add_highlight_annot(inst)
                        highlight.update()
                        modified = True
        
        if modified:
            # Save highlighted PDF in the same directory as the original
            output_name = pdf_file.parent / (pdf_file.stem + "_HIGHLIGHTED.pdf")
            doc.save(output_name, garbage=4, deflate=True)
            print(f"      ✨ Highlighted PDF saved: {output_name.name}")
        
        doc.close()
    
    print(f"   ✅ Completed scanning {source_name}")

# ============================================
# SCAN BOTH DIRECTORIES
# ============================================

results = []

# Scan Airbnb directory
scan_pdfs_in_directory(AIRBNB_DIR, results, "AirBNB")

# Scan VRBO directory
scan_pdfs_in_directory(VRBO_DIR, results, "VRBO")

# ============================================
# CREATE DATAFRAMES
# ============================================

df = pd.DataFrame(results)

if len(df) == 0:
    print("\n⚠️ No matches found. Check that PDFs are in the correct folders:")
    print(f"   - {AIRBNB_DIR}/ (and subfolders)")
    print(f"   - {VRBO_DIR}/ (and subfolders)")
    print("\n   Also verify that your PHRASES list contains the terms you want to search.")
else:
    print(f"\n📊 Total matches found: {len(df)}")

# Summary by phrase
phrase_summary = (
    df.groupby(["source", "phrase"])
      .size()
      .reset_index(name="match_count")
      .sort_values(["source", "match_count"], ascending=[True, False])
)

# Summary by file
file_summary = (
    df.groupby(["source", "file"])
      .size()
      .reset_index(name="total_matches")
      .sort_values(["source", "total_matches"], ascending=[True, False])
)

# Summary by page
page_summary = (
    df.groupby(["source", "file", "page"])
      .size()
      .reset_index(name="matches_on_page")
      .sort_values(["source", "file", "page"])
)

# ============================================
# EXPORT EXCEL FILES (One per source)
# ============================================

# Airbnb only
df_airbnb = df[df["source"] == "AirBNB"]
if len(df_airbnb) > 0:
    df_airbnb.to_excel("AirBNB_matches_detailed.xlsx", index=False)
    phrase_summary_airbnb = phrase_summary[phrase_summary["source"] == "AirBNB"]
    phrase_summary_airbnb.to_excel("AirBNB_phrase_summary.xlsx", index=False)
    print("\n📄 Airbnb Excel exports saved:")
    print("   - AirBNB_matches_detailed.xlsx")
    print("   - AirBNB_phrase_summary.xlsx")
else:
    print("\n⚠️ No Airbnb matches found to export.")

# VRBO only
df_vrbo = df[df["source"] == "VRBO"]
if len(df_vrbo) > 0:
    df_vrbo.to_excel("VRBO_matches_detailed.xlsx", index=False)
    phrase_summary_vrbo = phrase_summary[phrase_summary["source"] == "VRBO"]
    phrase_summary_vrbo.to_excel("VRBO_phrase_summary.xlsx", index=False)
    print("📄 VRBO Excel exports saved:")
    print("   - VRBO_matches_detailed.xlsx")
    print("   - VRBO_phrase_summary.xlsx")
else:
    print("⚠️ No VRBO matches found to export.")

# Combined summary
phrase_summary.to_excel("combined_phrase_summary.xlsx", index=False)
file_summary.to_excel("combined_file_summary.xlsx", index=False)
page_summary.to_excel("combined_page_summary.xlsx", index=False)
print("📄 Combined Excel exports saved:")
print("   - combined_phrase_summary.xlsx")
print("   - combined_file_summary.xlsx")
print("   - combined_page_summary.xlsx")

# ============================================
# GENERATE HTML REPORT
# ============================================

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Count total PDFs processed
total_pdfs_airbnb = len(list(AIRBNB_DIR.rglob("*.pdf")))
total_pdfs_vrbo = len(list(VRBO_DIR.rglob("*.pdf")))

html = f"""
<html>
<head>
<title>STR Rental Issue Report - Airbnb & VRBO</title>
<style>
    body {{ font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }}
    h1, h2 {{ color: #2c3e50; }}
    .container {{ max-width: 1400px; margin: 0 auto; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
    table {{ border-collapse: collapse; width: 100%; margin-bottom: 30px; font-size: 12px; }}
    th, td {{ border: 1px solid #ccc; padding: 8px; text-align: left; vertical-align: top; }}
    th {{ background-color: #2c3e50; color: white; }}
    tr:nth-child(even) {{ background-color: #fafafa; }}
    .summary-box {{ padding: 15px; background: #e8f4f8; border-left: 5px solid #3498db; margin-bottom: 30px; border-radius: 5px; }}
    .badge {{ display: inline-block; padding: 3px 8px; border-radius: 12px; font-size: 11px; font-weight: bold; }}
    .badge-airbnb {{ background: #ff5a5f; color: white; }}
    .badge-vrbo {{ background: #0f7b8a; color: white; }}
    hr {{ margin: 30px 0; }}
</style>
</head>
<body>
<div class="container">

<h1>🏠 STR Rental Issue Report</h1>
<p><strong>Generated:</strong> {timestamp}</p>

<div class="summary-box">
<h2>📊 Overview</h2>
<table style="width: auto;">
    <tr><th>Source</th><th>PDFs Scanned</th><th>Matches Found</th></tr>
    <tr><td><span class="badge badge-airbnb">AirBNB</span></td><td>{total_pdfs_airbnb}</td><td>{len(df_airbnb)}</td></tr>
    <tr><td><span class="badge badge-vrbo">VRBO</span></td><td>{total_pdfs_vrbo}</td><td>{len(df_vrbo)}</td></tr>
    <tr><td><strong>TOTAL</strong></td><td><strong>{total_pdfs_airbnb + total_pdfs_vrbo}</strong></td><td><strong>{len(df)}</strong></td></tr>
</table>
<p><strong>Phrases Searched:</strong> {len(PHRASES)} terms covering pests, cleanliness, parties, maintenance, host issues, and neighbor concerns.</p>
</div>

<h2>📈 Phrase Summary by Source</h2>
{phrase_summary.to_html(index=False)}

<hr>

<h2>📁 File Summary (Matches per File)</h2>
{file_summary.to_html(index=False)}

<hr>

<h2>📄 Detailed Matches</h2>
{df.to_html(index=False)}

<hr>

<h2>📍 Page Hotspots (Matches per Page)</h2>
{page_summary.to_html(index=False)}

</div>
</body>
</html>
"""

with open("STR_Issue_Report.html", "w", encoding="utf-8") as f:
    f.write(html)

print("\n🌐 HTML report saved as: STR_Issue_Report.html")
print("   Open in browser → Print → Save as PDF")

# ============================================
# DISPLAY PREVIEW IN NOTEBOOK (if applicable)
# ============================================

if len(df) > 0:
    print("\n📋 Preview of phrase summary:")
    display(phrase_summary.head(20))
    print("\n📋 Preview of file summary:")
    display(file_summary)
    print("\n📋 Preview of detailed matches:")
    display(df.head(10))
else:
    print("\n⚠️ No matches found to display.")

In [ ]:
pip install pandas openpyxl pymupdf

In [ ]:
# ============================================
# STR RENTAL ISSUE DETECTOR - Optimized Root Words
# ============================================

# Install once if needed
# !pip install pdfplumber pandas openpyxl pymupdf

import pdfplumber
import pandas as pd
import re
from pathlib import Path
import warnings
from datetime import datetime

# Try to import fitz (pymupdf) with error handling
try:
    import fitz
    FITZ_AVAILABLE = True
except ImportError:
    FITZ_AVAILABLE = False
    print("⚠️ Warning: pymupdf not installed. PDF highlighting will be disabled.")
    print("   To enable highlighting, run: pip install pymupdf\n")

warnings.filterwarnings("ignore")

# ============================================
# OPTIMIZED PHRASE LIST - Root Words for Maximum Coverage
# ============================================
# Using root words to catch multiple variations with one match
# No duplicates - each phrase is unique and catches related terms

PHRASES = [
    # === PESTS (root words catch all variations) ===
    "roach",           # catches "roach", "cockroach", "roaches"
    "bug",             # catches "bug", "bugs", "bed bug", "bedbugs"
    "infest",          # catches "infestation", "infested", "infesting"
    
    # === DIRT & FILTH (root words) ===
    "dirt",            # catches "dirt", "dirty", "dirtier"
    "dust",            # catches "dust", "dusty", "dustier"
    "filth",           # catches "filth", "filthy"
    "floor",           # catches "floor", "floors", "flooring"
    "sticky",          # catches "sticky", "stickiness"
    "not cleaned",     # exact phrase
    "cleaning issue",  # exact phrase
    "trash",           # catches "trash", "trashy", "trash bags"
    "garbage",         # catches "garbage"
    "rock",            # catches "rock", "rocks", "rocky"
    "leaf",            # catches "leaf", "leaves"
    "unvacuumed",      # exact word
    "stain",           # catches "stain", "stains", "stained"
    "freezer",         # catches "freezer", "freezers"
    "rug",             # catches "rug", "rugs"
    
    # === GUEST-CREATED PROBLEMS (root words) ===
    "bachelor",        # catches "bachelor", "bachelors", "bachelor's"
    "bachelorette",    # catches "bachelorette", "bachelorettes"
    "large group",     # exact phrase
    "sleeps up to",    # exact phrase
    "party",           # catches "party", "parties", "party's"
    "graduation",      # catches "graduation", "graduations"
    "tournament",      # catches "tournament", "tournaments"
    "super bowl",      # exact phrase
    "baby shower",     # exact phrase
    "entertainment home", # exact phrase
    "birthday",        # catches "birthday", "birthdays"
    "joint bachelor",  # exact phrase
    "group of",        # exact phrase
    
    # === HOST MAINTENANCE ISSUES (root words) ===
    "run down",        # exact phrase
    "tlc",             # exact phrase (TLC)
    "paint",           # catches "paint", "painted", "painting"
    "broken",          # catches "broken", "broke", "breaks"
    "not working",     # exact phrase
    "fire pit",        # exact phrase
    "spa",             # catches "spa", but careful - we'll use word boundary in regex
    "fridge",          # catches "fridge", "refrigerator"
    "cue ball",        # exact phrase
    "drain",           # catches "drain", "drains", "drained", "draining"
    "standing water",  # exact phrase
    "door sticks",     # exact phrase
    "latch",           # catches "latch", "latches", "latched", "latching"
    "wankered",        # exact word
    "ripped",          # catches "ripped", "rip", "rips"
    "wall damage",     # exact phrase
    "bed broken",      # exact phrase
    "bathroom fixtures", # exact phrase
    "towel rack",      # catches "towel rack", "towel racks"
    
    # === HOST COMMUNICATION FAILURES ===
    "door code",       # exact phrase
    "stuck in car",    # exact phrase
    "voicemail",       # catches "voicemail", "voice mail"
    "delayed check-in", # exact phrase
    "coffee pod",      # exact phrase
    "coffeepot",       # catches "coffeepot", "coffee pot"
    "grill",           # catches "grill", "grills", "grilling"
    "putt putt",       # exact phrase
    "listing accuracy", # exact phrase
    
    # === MISC ISSUES ===
    "asked if it would affect review", # exact phrase
    "hopeful i wouldn't leave a review", # exact phrase
    "quiet neighborhood", # exact phrase
]

# ============================================
# DIRECTORY SETUP
# ============================================

AIRBNB_DIR = Path("AirBNB")
VRBO_DIR = Path("VRBO")

AIRBNB_DIR.mkdir(exist_ok=True)
VRBO_DIR.mkdir(exist_ok=True)

# ============================================
# FUNCTION: SCAN PDFS RECURSIVELY
# ============================================

def scan_pdfs_in_directory(directory_path, results_list, source_name):
    """
    Recursively scan all PDFs in a directory and subdirectories.
    Highlights matches and returns results.
    """
    if not directory_path.exists():
        print(f"⚠️ Directory not found: {directory_path}")
        return
    
    # Find all PDF files recursively, skip previously highlighted ones
    pdf_files = list(directory_path.rglob("*.pdf"))
    pdf_files = [f for f in pdf_files if not f.stem.endswith("_HIGHLIGHTED")]
    
    print(f"\n📁 Scanning {source_name} directory: {directory_path}")
    print(f"   Found {len(pdf_files)} PDF(s) to process...")
    
    if len(pdf_files) == 0:
        print(f"   No PDFs found in {directory_path} or its subfolders.")
        return
    
    for pdf_file in pdf_files:
        print(f"   Processing: {pdf_file.name} (in {pdf_file.parent.name if pdf_file.parent != directory_path else 'root'})")
        
        try:
            # Extract text and find matches
            with pdfplumber.open(pdf_file) as pdf:
                for page_num, page in enumerate(pdf.pages, start=1):
                    text = page.extract_text()
                    if not text:
                        continue
                    
                    lines = text.split("\n")
                    
                    for line_num, line in enumerate(lines, start=1):
                        # Check each phrase
                        for phrase in PHRASES:
                            # Use regex with word boundaries for single words to avoid false matches
                            # For multi-word phrases, use simple search
                            if ' ' in phrase:
                                # Multi-word phrase - search as is
                                if re.search(re.escape(phrase), line, re.IGNORECASE):
                                    line_text = line.strip()
                                    if line_text.lower() == "savannah, georgia":
                                        line_text += " - Local Guest"
                                    results_list.append({
                                        "source": source_name,
                                        "file": str(pdf_file.relative_to(directory_path.parent)) if pdf_file.parent != directory_path else pdf_file.name,
                                        "page": page_num,
                                        "line_number": line_num,
                                        "phrase": phrase,
                                        "line_text": line_text
                                    })
                            else:
                                # Single word - use word boundaries to avoid false matches
                                # Special handling for "spa" to avoid matching "space"
                                if phrase == "spa":
                                    pattern = r'\bspa\b'
                                else:
                                    pattern = r'\b' + re.escape(phrase) + r'\b'
                                
                                if re.search(pattern, line, re.IGNORECASE):
                                    line_text = line.strip()
                                    if line_text.lower() == "savannah, georgia":
                                        line_text += " - Local Guest"
                                    results_list.append({
                                        "source": source_name,
                                        "file": str(pdf_file.relative_to(directory_path.parent)) if pdf_file.parent != directory_path else pdf_file.name,
                                        "page": page_num,
                                        "line_number": line_num,
                                        "phrase": phrase,
                                        "line_text": line_text
                                    })
        except Exception as e:
            print(f"      ⚠️ Error reading {pdf_file.name}: {e}")
            continue
        
        # Highlight matches in PDF (only if fitz is available)
        if FITZ_AVAILABLE:
            try:
                doc = fitz.open(pdf_file)
                modified = False
                
                for page in doc:
                    page_text = page.get_text("text")
                    
                    for phrase in PHRASES:
                        if ' ' in phrase:
                            pattern = re.escape(phrase)
                        else:
                            if phrase == "spa":
                                pattern = r'\bspa\b'
                            else:
                                pattern = r'\b' + re.escape(phrase) + r'\b'
                        
                        for match in re.finditer(pattern, page_text, re.IGNORECASE):
                            matched_text = match.group(0)
                            text_instances = page.search_for(matched_text)
                            
                            for inst in text_instances:
                                highlight = page.add_highlight_annot(inst)
                                highlight.update()
                                modified = True
                
                if modified:
                    output_name = pdf_file.parent / (pdf_file.stem + "_HIGHLIGHTED.pdf")
                    doc.save(output_name, garbage=4, deflate=True)
                    print(f"      ✨ Highlighted PDF saved: {output_name.name}")
                
                doc.close()
            except Exception as e:
                print(f"      ⚠️ Error highlighting {pdf_file.name}: {e}")
        else:
            pass
    
    print(f"   ✅ Completed scanning {source_name}")

# ============================================
# SCAN BOTH DIRECTORIES
# ============================================

print("=" * 60)
print("STR RENTAL ISSUE DETECTOR - Optimized Root Words")
print("=" * 60)
print(f"\n📂 Looking for PDFs in:")
print(f"   - {AIRBNB_DIR.absolute()}/")
print(f"   - {VRBO_DIR.absolute()}/")
print(f"\n🔍 Searching for {len(PHRASES)} optimized root-word phrases...")
print("   (e.g., 'roach' catches both 'roach' AND 'cockroach')")
if not FITZ_AVAILABLE:
    print("⚠️ PDF highlighting disabled (install pymupdf to enable)")
print()

results = []

scan_pdfs_in_directory(AIRBNB_DIR, results, "AirBNB")
scan_pdfs_in_directory(VRBO_DIR, results, "VRBO")

# ============================================
# CREATE DATAFRAMES
# ============================================

df = pd.DataFrame(results)

if len(df) == 0:
    print("\n" + "=" * 60)
    print("⚠️ NO MATCHES FOUND")
    print("=" * 60)
    print("\nPossible reasons:")
    print("   1. No PDF files in the AirBNB/ or VRBO/ folders")
    print("   2. PDFs are image-based (not text searchable)")
    print("   3. Keywords don't match the content")
    print("\nTo fix:")
    print("   - Place PDFs in the correct folders")
    print("   - Ensure PDFs have selectable text (not scanned images)")
    print("   - Add more keywords to the PHRASES list")
    print(f"\n📁 AirBNB folder: {AIRBNB_DIR.absolute()}")
    print(f"📁 VRBO folder: {VRBO_DIR.absolute()}")
else:
    print(f"\n📊 Total matches found: {len(df)}")

# Summary by phrase
if len(df) > 0:
    phrase_summary = (
        df.groupby(["source", "phrase"])
          .size()
          .reset_index(name="match_count")
          .sort_values(["source", "match_count"], ascending=[True, False])
    )

    # Summary by file
    file_summary = (
        df.groupby(["source", "file"])
          .size()
          .reset_index(name="total_matches")
          .sort_values(["source", "total_matches"], ascending=[True, False])
    )

    # Summary by page
    page_summary = (
        df.groupby(["source", "file", "page"])
          .size()
          .reset_index(name="matches_on_page")
          .sort_values(["source", "file", "page"])
    )

    # ============================================
    # EXPORT EXCEL FILES
    # ============================================

    # Airbnb only
    df_airbnb = df[df["source"] == "AirBNB"]
    if len(df_airbnb) > 0:
        df_airbnb.to_excel("AirBNB_matches_detailed.xlsx", index=False)
        phrase_summary_airbnb = phrase_summary[phrase_summary["source"] == "AirBNB"]
        phrase_summary_airbnb.to_excel("AirBNB_phrase_summary.xlsx", index=False)
        print("\n📄 Airbnb Excel exports saved:")
        print("   - AirBNB_matches_detailed.xlsx")
        print("   - AirBNB_phrase_summary.xlsx")
    else:
        print("\n⚠️ No Airbnb matches found to export.")

    # VRBO only
    df_vrbo = df[df["source"] == "VRBO"]
    if len(df_vrbo) > 0:
        df_vrbo.to_excel("VRBO_matches_detailed.xlsx", index=False)
        phrase_summary_vrbo = phrase_summary[phrase_summary["source"] == "VRBO"]
        phrase_summary_vrbo.to_excel("VRBO_phrase_summary.xlsx", index=False)
        print("📄 VRBO Excel exports saved:")
        print("   - VRBO_matches_detailed.xlsx")
        print("   - VRBO_phrase_summary.xlsx")
    else:
        print("⚠️ No VRBO matches found to export.")

    # Combined exports
    phrase_summary.to_excel("combined_phrase_summary.xlsx", index=False)
    file_summary.to_excel("combined_file_summary.xlsx", index=False)
    page_summary.to_excel("combined_page_summary.xlsx", index=False)
    print("📄 Combined Excel exports saved:")
    print("   - combined_phrase_summary.xlsx")
    print("   - combined_file_summary.xlsx")
    print("   - combined_page_summary.xlsx")

    # ============================================
    # GENERATE HTML REPORT
    # ============================================

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Count total PDFs processed
    total_pdfs_airbnb = len(list(AIRBNB_DIR.rglob("*.pdf")))
    total_pdfs_vrbo = len(list(VRBO_DIR.rglob("*.pdf")))

    html = f"""
    <html>
    <head>
    <title>STR Rental Issue Report - Airbnb & VRBO</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }}
        h1, h2 {{ color: #2c3e50; }}
        .container {{ max-width: 1400px; margin: 0 auto; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
        table {{ border-collapse: collapse; width: 100%; margin-bottom: 30px; font-size: 12px; }}
        th, td {{ border: 1px solid #ccc; padding: 8px; text-align: left; vertical-align: top; }}
        th {{ background-color: #2c3e50; color: white; }}
        tr:nth-child(even) {{ background-color: #fafafa; }}
        .summary-box {{ padding: 15px; background: #e8f4f8; border-left: 5px solid #3498db; margin-bottom: 30px; border-radius: 5px; }}
        .badge {{ display: inline-block; padding: 3px 8px; border-radius: 12px; font-size: 11px; font-weight: bold; }}
        .badge-airbnb {{ background: #ff5a5f; color: white; }}
        .badge-vrbo {{ background: #0f7b8a; color: white; }}
        hr {{ margin: 30px 0; }}
    </style>
    </head>
    <body>
    <div class="container">

    <h1>🏠 STR Rental Issue Report</h1>
    <p><strong>Generated:</strong> {timestamp}</p>
    <p><strong>Note:</strong> Using root-word optimization - "roach" catches both "roach" AND "cockroach"</p>

    <div class="summary-box">
    <h2>📊 Overview</h2>
    <table style="width: auto;">
        <tr><th>Source</th><th>PDFs Scanned</th><th>Matches Found</th> </tr>
        <tr><td><span class="badge badge-airbnb">AirBNB</span></td><td>{total_pdfs_airbnb}</td><td>{len(df_airbnb) if len(df_airbnb) > 0 else 0}</td> </tr>
        <tr><td><span class="badge badge-vrbo">VRBO</span></td><td>{total_pdfs_vrbo}</td><td>{len(df_vrbo) if len(df_vrbo) > 0 else 0}</td> </tr>
        <tr><td><strong>TOTAL</strong></td><td><strong>{total_pdfs_airbnb + total_pdfs_vrbo}</strong></td><td><strong>{len(df)}</strong></td> </tr>
     </table>
    <p><strong>Phrases Searched:</strong> {len(PHRASES)} optimized root-word terms covering pests, cleanliness, parties, maintenance, host issues, and neighbor concerns.</p>
    </div>

    <h2>📈 Phrase Summary by Source</h2>
    {phrase_summary.to_html(index=False)}

    <hr>

    <h2>📁 File Summary (Matches per File)</h2>
    {file_summary.to_html(index=False)}

    <hr>

    <h2>📄 Detailed Matches</h2>
    {df.to_html(index=False)}

    <hr>

    <h2>📍 Page Hotspots (Matches per Page)</h2>
    {page_summary.to_html(index=False)}

    </div>
    </body>
    </html>
    """

    with open("STR_Issue_Report.html", "w", encoding="utf-8") as f:
        f.write(html)

    print("\n🌐 HTML report saved as: STR_Issue_Report.html")

    # Display preview
    print("\n📋 Preview of phrase summary (first 20):")
    display(phrase_summary.head(20))

In [15]:
# ============================================
# STR RENTAL ISSUE DETECTOR - Optimized Root Words (FIXED HTML)
# ============================================

# Install once if needed
# !pip install pdfplumber pandas openpyxl pymupdf

import pdfplumber
import pandas as pd
import re
from pathlib import Path
import warnings
from datetime import datetime

# Try to import fitz (pymupdf) with error handling
try:
    import fitz
    FITZ_AVAILABLE = True
except ImportError:
    FITZ_AVAILABLE = False
    print("⚠️ Warning: pymupdf not installed. PDF highlighting will be disabled.")
    print("   To enable highlighting, run: pip install pymupdf\n")

warnings.filterwarnings("ignore")

# ============================================
# OPTIMIZED PHRASE LIST - Root Words for Maximum Coverage
# ============================================

PHRASES = [
    # === PESTS ===
    "roach",           # catches "roach", "cockroach", "roaches"
    "bug",             # catches "bug", "bugs", "bed bug", "bedbugs"
    "infest",          # catches "infestation", "infested", "infesting"
    
    # === DIRT & FILTH ===
    "dirt",
    "dust",
    "filth",
    "floor",
    "sticky",
    "not cleaned",
    "cleaning issue",
    "trash",
    "garbage",
    "rock",
    "leaf",
    "vacuum",
    "stain",
    
    # === GUEST-CREATED PROBLEMS ===
    "bachelor",
    "group",
    "large group",
    "sleeps up to",
    "party",
    "fun"
    "graduation",
    "tournament",
    "super bowl",
    "baby shower",
    "entertainment home",
    "birthday",
    "joint bachelor",
    "group of",
    
    # === Attracting parties with ammenities ===    
    "putt putt",
    "golf",
    "mini putt",
    "pickle ball",
    "projector",
    "tv",
    "fun ammenities",
    "fun house",
    "party house",
    "entertain",
    # === HOST MAINTENANCE ISSUES ===
    "run down",
    "tlc",
    "paint",
    "broken",
    "not work",
    "fire pit",
    "spa",
    "fridge",
    "cue ball",
    "drain",
    "standing water",
    "door sticks",
    "latch",
    "wankered",
    "ripped",
    "wall damage",
    "bed broken",
    "bathroom fixtures",
    "towel rack",
    "Disliked:"
    
    # === HOST COMMUNICATION FAILURES ===
    "door code",
    "stuck in car",
    "delayed check-in",

    # === MISC ISSUES ===
    "affect review",
    "leave a review",
]

# ============================================
# HELPER FUNCTION: Count only original PDFs (exclude _HIGHLIGHTED)
# ============================================

def count_original_pdfs(directory_path):
    """Count PDFs that don't have _HIGHLIGHTED in the filename"""
    if not directory_path.exists():
        return 0
    all_pdfs = list(directory_path.rglob("*.pdf"))
    original_pdfs = [f for f in all_pdfs if not f.stem.endswith("_HIGHLIGHTED")]
    return len(original_pdfs)

# ============================================
# DIRECTORY SETUP
# ============================================

AIRBNB_DIR = Path("AirBNB")
VRBO_DIR = Path("VRBO")

AIRBNB_DIR.mkdir(exist_ok=True)
VRBO_DIR.mkdir(exist_ok=True)

# ============================================
# FUNCTION: SCAN PDFS RECURSIVELY
# ============================================

def scan_pdfs_in_directory(directory_path, results_list, source_name):
    """
    Recursively scan all PDFs in a directory and subdirectories.
    Highlights matches and returns results.
    """
    if not directory_path.exists():
        print(f"⚠️ Directory not found: {directory_path}")
        return 0
    
    # Find all PDF files recursively, skip previously highlighted ones
    all_pdfs = list(directory_path.rglob("*.pdf"))
    pdf_files = [f for f in all_pdfs if not f.stem.endswith("_HIGHLIGHTED")]
    
    print(f"\n📁 Scanning {source_name} directory: {directory_path}")
    print(f"   Found {len(pdf_files)} original PDF(s) to process...")
    
    if len(pdf_files) == 0:
        print(f"   No PDFs found in {directory_path} or its subfolders.")
        return 0
    
    for pdf_file in pdf_files:
        print(f"   Processing: {pdf_file.name} (in {pdf_file.parent.name if pdf_file.parent != directory_path else 'root'})")
        
        try:
            # Extract text and find matches
            with pdfplumber.open(pdf_file) as pdf:
                for page_num, page in enumerate(pdf.pages, start=1):
                    text = page.extract_text()
                    if not text:
                        continue
                    
                    lines = text.split("\n")
                    
                    for line_num, line in enumerate(lines, start=1):
                        # Check each phrase
                        for phrase in PHRASES:
                            # Use regex with word boundaries for single words to avoid false matches
                            if ' ' in phrase:
                                # Multi-word phrase - search as is
                                if re.search(re.escape(phrase), line, re.IGNORECASE):
                                    line_text = line.strip()
                                    if line_text.lower() == "savannah, georgia":
                                        line_text += " - Local Guest"
                                    results_list.append({
                                        "source": source_name,
                                        "file": str(pdf_file.relative_to(directory_path.parent)) if pdf_file.parent != directory_path else pdf_file.name,
                                        "page": page_num,
                                        "line_number": line_num,
                                        "phrase": phrase,
                                        "line_text": line_text
                                    })
                            else:
                                # Single word - use word boundaries
                                if phrase == "spa":
                                    pattern = r'\bspa\b'
                                else:
                                    pattern = r'\b' + re.escape(phrase) + r'\b'
                                
                                if re.search(pattern, line, re.IGNORECASE):
                                    line_text = line.strip()
                                    if line_text.lower() == "savannah, georgia":
                                        line_text += " - Local Guest"
                                    results_list.append({
                                        "source": source_name,
                                        "file": str(pdf_file.relative_to(directory_path.parent)) if pdf_file.parent != directory_path else pdf_file.name,
                                        "page": page_num,
                                        "line_number": line_num,
                                        "phrase": phrase,
                                        "line_text": line_text
                                    })
        except Exception as e:
            print(f"      ⚠️ Error reading {pdf_file.name}: {e}")
            continue
        
        # Highlight matches in PDF (only if fitz is available)
        if FITZ_AVAILABLE:
            try:
                doc = fitz.open(pdf_file)
                modified = False
                
                for page in doc:
                    page_text = page.get_text("text")
                    
                    for phrase in PHRASES:
                        if ' ' in phrase:
                            pattern = re.escape(phrase)
                        else:
                            if phrase == "spa":
                                pattern = r'\bspa\b'
                            else:
                                pattern = r'\b' + re.escape(phrase) + r'\b'
                        
                        for match in re.finditer(pattern, page_text, re.IGNORECASE):
                            matched_text = match.group(0)
                            text_instances = page.search_for(matched_text)
                            
                            for inst in text_instances:
                                highlight = page.add_highlight_annot(inst)
                                highlight.update()
                                modified = True
                
                if modified:
                    output_name = pdf_file.parent / (pdf_file.stem + "_HIGHLIGHTED.pdf")
                    doc.save(output_name, garbage=4, deflate=True)
                    print(f"      ✨ Highlighted PDF saved: {output_name.name}")
                
                doc.close()
            except Exception as e:
                print(f"      ⚠️ Error highlighting {pdf_file.name}: {e}")
        
    print(f"   ✅ Completed scanning {source_name}")
    return len(pdf_files)

# ============================================
# SCAN BOTH DIRECTORIES
# ============================================

print("=" * 60)
print("STR RENTAL ISSUE DETECTOR - Optimized Root Words")
print("=" * 60)
print(f"\n📂 Looking for PDFs in:")
print(f"   - {AIRBNB_DIR.absolute()}/")
print(f"   - {VRBO_DIR.absolute()}/")
print(f"\n🔍 Searching for {len(PHRASES)} optimized root-word phrases...")
print("   (e.g., 'roach' catches both 'roach' AND 'cockroach')")
if not FITZ_AVAILABLE:
    print("⚠️ PDF highlighting disabled (install pymupdf to enable)")
print()

results = []

# Scan and get counts of original PDFs
airbnb_count = scan_pdfs_in_directory(AIRBNB_DIR, results, "AirBNB")
vrbo_count = scan_pdfs_in_directory(VRBO_DIR, results, "VRBO")

# ============================================
# CREATE DATAFRAMES
# ============================================

df = pd.DataFrame(results)

if len(df) == 0:
    print("\n" + "=" * 60)
    print("⚠️ NO MATCHES FOUND")
    print("=" * 60)
    print("\nPossible reasons:")
    print("   1. No PDF files in the AirBNB/ or VRBO/ folders")
    print("   2. PDFs are image-based (not text searchable)")
    print("   3. Keywords don't match the content")
    print("\nTo fix:")
    print("   - Place PDFs in the correct folders")
    print("   - Ensure PDFs have selectable text (not scanned images)")
    print("   - Add more keywords to the PHRASES list")
    print(f"\n📁 AirBNB folder: {AIRBNB_DIR.absolute()}")
    print(f"📁 VRBO folder: {VRBO_DIR.absolute()}")
else:
    print(f"\n📊 Total matches found: {len(df)}")

# Summary by phrase
if len(df) > 0:
    phrase_summary = (
        df.groupby(["source", "phrase"])
          .size()
          .reset_index(name="match_count")
          .sort_values(["source", "match_count"], ascending=[True, False])
    )

    # Summary by file
    file_summary = (
        df.groupby(["source", "file"])
          .size()
          .reset_index(name="total_matches")
          .sort_values(["source", "total_matches"], ascending=[True, False])
    )

    # Summary by page
    page_summary = (
        df.groupby(["source", "file", "page"])
          .size()
          .reset_index(name="matches_on_page")
          .sort_values(["source", "file", "page"])
    )

    # ============================================
    # EXPORT EXCEL FILES
    # ============================================

    # Airbnb only
    df_airbnb = df[df["source"] == "AirBNB"]
    if len(df_airbnb) > 0:
        df_airbnb.to_excel("AirBNB_matches_detailed.xlsx", index=False)
        phrase_summary_airbnb = phrase_summary[phrase_summary["source"] == "AirBNB"]
        phrase_summary_airbnb.to_excel("AirBNB_phrase_summary.xlsx", index=False)
        print("\n📄 Airbnb Excel exports saved:")
        print("   - AirBNB_matches_detailed.xlsx")
        print("   - AirBNB_phrase_summary.xlsx")
    else:
        print("\n⚠️ No Airbnb matches found to export.")

    # VRBO only
    df_vrbo = df[df["source"] == "VRBO"]
    if len(df_vrbo) > 0:
        df_vrbo.to_excel("VRBO_matches_detailed.xlsx", index=False)
        phrase_summary_vrbo = phrase_summary[phrase_summary["source"] == "VRBO"]
        phrase_summary_vrbo.to_excel("VRBO_phrase_summary.xlsx", index=False)
        print("📄 VRBO Excel exports saved:")
        print("   - VRBO_matches_detailed.xlsx")
        print("   - VRBO_phrase_summary.xlsx")
    else:
        print("⚠️ No VRBO matches found to export.")

    # Combined exports
    phrase_summary.to_excel("combined_phrase_summary.xlsx", index=False)
    file_summary.to_excel("combined_file_summary.xlsx", index=False)
    page_summary.to_excel("combined_page_summary.xlsx", index=False)
    print("📄 Combined Excel exports saved:")
    print("   - combined_phrase_summary.xlsx")
    print("   - combined_file_summary.xlsx")
    print("   - combined_page_summary.xlsx")

    # ============================================
    # GENERATE HTML REPORT (FIXED)
    # ============================================

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Count ONLY original PDFs (exclude _HIGHLIGHTED)
    total_pdfs_airbnb = count_original_pdfs(AIRBNB_DIR)
    total_pdfs_vrbo = count_original_pdfs(VRBO_DIR)
    
    # Get match counts
    airbnb_match_count = len(df_airbnb) if len(df_airbnb) > 0 else 0
    vrbo_match_count = len(df_vrbo) if len(df_vrbo) > 0 else 0

    # Create overview table HTML
    overview_html = f"""
    <table style="width: auto; border-collapse: collapse;">
        <thead>
            <tr>
                <th style="border: 1px solid #ccc; padding: 8px; background-color: #2c3e50; color: white;">Source</th>
                <th style="border: 1px solid #ccc; padding: 8px; background-color: #2c3e50; color: white;">Original PDFs</th>
                <th style="border: 1px solid #ccc; padding: 8px; background-color: #2c3e50; color: white;">Matches Found</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td style="border: 1px solid #ccc; padding: 8px;"><span class="badge badge-airbnb">AirBNB</span></td>
                <td style="border: 1px solid #ccc; padding: 8px;">{total_pdfs_airbnb}</td>
                <td style="border: 1px solid #ccc; padding: 8px;">{airbnb_match_count}</td>
            </tr>
            <tr>
                <td style="border: 1px solid #ccc; padding: 8px;"><span class="badge badge-vrbo">VRBO</span></td>
                <td style="border: 1px solid #ccc; padding: 8px;">{total_pdfs_vrbo}</td>
                <td style="border: 1px solid #ccc; padding: 8px;">{vrbo_match_count}</td>
            </tr>
            <tr style="background-color: #f2f2f2;">
                <td style="border: 1px solid #ccc; padding: 8px;"><strong>TOTAL</strong></td>
                <td style="border: 1px solid #ccc; padding: 8px;"><strong>{total_pdfs_airbnb + total_pdfs_vrbo}</strong></td>
                <td style="border: 1px solid #ccc; padding: 8px;"><strong>{len(df)}</strong></td>
            </tr>
        </tbody>
    </table>
    """

    html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>STR Rental Issue Report - Airbnb & VRBO</title>
<style>
    body {{ font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }}
    h1, h2 {{ color: #2c3e50; }}
    .container {{ max-width: 1400px; margin: 0 auto; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
    table {{ border-collapse: collapse; width: 100%; margin-bottom: 30px; font-size: 12px; }}
    th, td {{ border: 1px solid #ccc; padding: 8px; text-align: left; vertical-align: top; }}
    th {{ background-color: #2c3e50; color: white; }}
    tr:nth-child(even) {{ background-color: #fafafa; }}
    .summary-box {{ padding: 15px; background: #e8f4f8; border-left: 5px solid #3498db; margin-bottom: 30px; border-radius: 5px; }}
    .badge {{ display: inline-block; padding: 3px 8px; border-radius: 12px; font-size: 11px; font-weight: bold; }}
    .badge-airbnb {{ background: #ff5a5f; color: white; }}
    .badge-vrbo {{ background: #0f7b8a; color: white; }}
    hr {{ margin: 30px 0; }}
</style>
</head>
<body>
<div class="container">

<h1>🏠 STR Rental Issue Report</h1>
<p><strong>Generated:</strong> {timestamp}</p>
<p><strong>Note:</strong> Using root-word optimization - "roach" catches both "roach" AND "cockroach"</p>

<div class="summary-box">
<h2>📊 Overview</h2>
{overview_html}
<p><strong>Phrases Searched:</strong> {len(PHRASES)} optimized root-word terms covering pests, cleanliness, parties, maintenance, host issues, and neighbor concerns.</p>
<p><strong>Note:</strong> Highlighted PDFs (_HIGHLIGHTED) are excluded from counts.</p>
</div>

<h2>📈 Phrase Summary by Source</h2>
{phrase_summary.to_html(index=False)}

<hr>

<h2>📁 File Summary (Matches per File)</h2>
{file_summary.to_html(index=False)}

<hr>

<h2>📄 Detailed Matches</h2>
{df.to_html(index=False, max_rows=1000)}

<hr>

<h2>📍 Page Hotspots (Matches per Page)</h2>
{page_summary.to_html(index=False)}

</div>
</body>
</html>
"""

    with open("STR_Issue_Report.html", "w", encoding="utf-8") as f:
        f.write(html)

    print("\n🌐 HTML report saved as: STR_Issue_Report.html")
    print("   Open in browser → Print → Save as PDF")

    # Display preview
    print("\n📋 Preview of phrase summary (first 20):")
    display(phrase_summary.head(20))

STR RENTAL ISSUE DETECTOR - Optimized Root Words

📂 Looking for PDFs in:
   - c:\Users\micha\stvr-info-page\owners\cozy_corner_rentals\review_pdfs\AirBNB/
   - c:\Users\micha\stvr-info-page\owners\cozy_corner_rentals\review_pdfs\VRBO/

🔍 Searching for 63 optimized root-word phrases...
   (e.g., 'roach' catches both 'roach' AND 'cockroach')


📁 Scanning AirBNB directory: AirBNB
   Found 3 original PDF(s) to process...
   Processing: Whitefield-airbnb-reviews-2-26-26.pdf (in root)


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


      ✨ Highlighted PDF saved: Whitefield-airbnb-reviews-2-26-26_HIGHLIGHTED.pdf
   Processing: backshell -airbnb-reviews-3-28-2026.pdf (in march 28)


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


      ✨ Highlighted PDF saved: backshell -airbnb-reviews-3-28-2026_HIGHLIGHTED.pdf
   Processing: halcyon-airbnb-reviews-3-28-2026.pdf (in march 28)
      ✨ Highlighted PDF saved: halcyon-airbnb-reviews-3-28-2026_HIGHLIGHTED.pdf
   ✅ Completed scanning AirBNB

📁 Scanning VRBO directory: VRBO
   Found 3 original PDF(s) to process...
   Processing: backshell-VRBO-reviews-3-28-2026.pdf (in march 28 2026)
      ✨ Highlighted PDF saved: backshell-VRBO-reviews-3-28-2026_HIGHLIGHTED.pdf
   Processing: halcyon-VRBO-reviews-3-28-2026.pdf (in march 28 2026)
      ✨ Highlighted PDF saved: halcyon-VRBO-reviews-3-28-2026_HIGHLIGHTED.pdf
   Processing: whitefield-VRBO-reviews-3-28-2026.pdf (in march 28 2026)
      ✨ Highlighted PDF saved: whitefield-VRBO-reviews-3-28-2026_HIGHLIGHTED.pdf
   ✅ Completed scanning VRBO

📊 Total matches found: 252

📄 Airbnb Excel exports saved:
   - AirBNB_matches_detailed.xlsx
   - AirBNB_phrase_summary.xlsx
📄 VRBO Excel exports saved:
   - VRBO_matches_detailed.xlsx
 

,source,phrase,match_count
13,AirBNB,group,123
20,AirBNB,party,11
23,AirBNB,projector,11
31,AirBNB,tv,7
9,AirBNB,floor,6
2,AirBNB,birthday,5
14,AirBNB,group of,5
0,AirBNB,bachelor,4
3,AirBNB,broken,4
12,AirBNB,golf,4
